# 워크플로우가 충분한 조건

다섯 가지 질문으로 중간 프로젝트에 필요한 자율성 수준을 판단합니다.

## 실행 환경 준비

점수와 비용은 설계를 비교하기 위한 교육용 고정값입니다. 모델 호출, 실제 청구, 품질 측정을 수행하지 않습니다.

In [1]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
from pathlib import Path
import sys

start = Path.cwd().resolve()
candidates = (start, *start.parents)
ROOT = next(path for path in candidates if (path / "labs/part5/ch17-agent-overview").is_dir())
LAB = ROOT / "labs/part5/ch17-agent-overview"
print("Python:", sys.version.split()[0], "| 실습 폴더:", LAB.relative_to(ROOT))

Python: 3.12.13 | 실습 폴더: labs/part5/ch17-agent-overview


## 진단 기준

경로를 미리 적을 수 있고 실패 처리와 승인 지점이 명확할수록 워크플로우가 적합합니다.

In [2]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
criteria = [
    ("단계를 미리 나열할 수 있는가", 2),
    ("입력 유형을 안정적으로 구분할 수 있는가", 1),
    ("실패 때 실행할 대체 경로가 정해져 있는가", 1),
    ("외부 쓰기 전에 승인 지점을 둘 수 있는가", 1),
    ("같은 입력에 같은 절차가 중요한가", 1),
]
print("워크플로우 적합 최대점:", sum(weight for _, weight in criteria))

워크플로우 적합 최대점: 6


## 두 프로젝트 비교

값은 가상 수업 사례에 대해 미리 정한 진단 응답입니다.

In [3]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
projects = {
    "사내 규정 질의응답": [True, True, True, True, True],
    "낯선 저장소 장애 수정": [False, False, False, True, False],
}

def workflow_score(answers: list[bool]) -> int:
    return sum(weight for (_, weight), answer in zip(criteria, answers) if answer)

for name, answers in projects.items():
    score = workflow_score(answers)
    recommendation = "워크플로우 우선" if score >= 4 else "제한된 에이전트 검토"
    print(name, score, "/ 6", recommendation)
assert workflow_score(projects["사내 규정 질의응답"]) == 6
assert workflow_score(projects["낯선 저장소 장애 수정"]) == 1

사내 규정 질의응답 6 / 6 워크플로우 우선
낯선 저장소 장애 수정 1 / 6 제한된 에이전트 검토


## 불필요한 자율성의 교육용 비용

규정 질의응답에 모델 결정 단계를 두 개 추가했다고 가정합니다. 호출 수와 시간은 실제 측정값이 아닙니다.

In [4]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
fixed_scenario = {"calls": 2, "latency_s": 2.4, "cost_units": 2}
agentic_scenario = {"calls": 4, "latency_s": 5.9, "cost_units": 5}
for key in fixed_scenario:
    delta = agentic_scenario[key] - fixed_scenario[key]
    shown = f"{delta:.1f}" if isinstance(delta, float) else delta
    print(key, "교육용 차이:", shown)
print("품질 향상은 이 데이터로 측정하지 않았습니다.")

calls 교육용 차이: 2
latency_s 교육용 차이: 3.5
cost_units 교육용 차이: 3
품질 향상은 이 데이터로 측정하지 않았습니다.


## 민감도 확인

임계값 변화에 따라 추천이 달라지는지 확인합니다.

In [5]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
for threshold in (3, 4, 5):
    labels = {name: workflow_score(value) >= threshold for name, value in projects.items()}
    print("임계값", threshold, labels)
print("경계 점수는 팀 검토가 필요하며 자동 결정 규칙으로 사용하지 않습니다.")
assert agentic_scenario["calls"] - fixed_scenario["calls"] == 2
assert agentic_scenario["cost_units"] - fixed_scenario["cost_units"] == 3

임계값 3 {'사내 규정 질의응답': True, '낯선 저장소 장애 수정': False}
임계값 4 {'사내 규정 질의응답': True, '낯선 저장소 장애 수정': False}
임계값 5 {'사내 규정 질의응답': True, '낯선 저장소 장애 수정': False}
경계 점수는 팀 검토가 필요하며 자동 결정 규칙으로 사용하지 않습니다.
